In [ ]:
import numpy as np
import pandas as pd

In [ ]:
rating = pd.read_csv("/data/zyang/datasets/ml-1m/ratings.dat",header=None, sep='::', names=['uid','iid','rating','timestamp'],engine='python')
item_info = pd.read_csv("/data/zyang/datasets/ml-1m/movies.dat",sep='::',encoding='latin-1',engine='python',names=['iid','title','genres'])
user_info = pd.read_csv("/data/zyang/datasets/ml-1m/users.dat",sep='::',encoding='latin-1',engine='python',names=['uid','gender','age','occupation','zipcode'])

In [ ]:
item_info.head()

In [ ]:
user_info.head(2)

In [ ]:
rating_ = pd.merge(rating,item_info,on='iid',how='inner')

In [ ]:
rating_.head(2)

In [ ]:
date_min = pd.to_datetime(rating_.timestamp,unit='s').min() 
date_max = pd.to_datetime(rating_.timestamp,unit='s').max()
date_min,date_max

In [ ]:
(date_max.year-date_min.year)*12+date_max.month

In [ ]:
rating_['time'] = pd.to_datetime(rating_.timestamp,unit='s').map(lambda x: (x.year-date_min.year)*12+x.month)

In [ ]:
rating_.head(2)

In [ ]:
rating_['time'] = rating_['time'] - rating_['time'].min()

In [ ]:
np.sort(rating_.time.unique())

In [ ]:
rating_.head(2)

In [ ]:
rating_.groupby('time').agg({'rating':'count'}).reset_index().plot(x='time',kind='bar')

In [ ]:
rating_.head(2)

In [ ]:
train_slot = list(range(14,24))
valid_slot = list(range(24,29))
test_slot = list(range(29,34))

In [ ]:
rating_['label'] = rating_['rating'].apply(lambda x: 1 if x>=4 else 0)
rating_.head(5)

In [ ]:
rating_.label.describe()

In [ ]:
rating_.uid.min(),rating_.uid.max(),rating_.iid.min(),rating_.iid.max()

## split by time order

In [ ]:
rating_train = rating_[rating_['time'].isin(train_slot)].copy()
rating_valid = rating_[rating_['time'].isin(valid_slot)].copy()
rating_test = rating_[rating_['time'].isin(test_slot)].copy()

In [ ]:
rating_train.shape, rating_valid.shape, rating_test.shape

In [ ]:
rating_valid_f = rating_valid
rating_test_f = rating_test

In [ ]:
def filter_cold_start(train,valid,test):
    train_user = train.uid.unique()
    train_item = train.iid.unique()
    valid = valid[valid['uid'].isin(train_user)]
    test = test[test['uid'].isin(train_user)]
    valid = valid[valid['iid'].isin(train_item)]
    test = test[test['iid'].isin(train_item)]
    return valid, test

In [ ]:
# rating_valid_f, rating_test_f = filter_cold_start(rating_train,rating_valid,rating_test)
# rating_valid_f.shape, rating_test_f.shape

In [ ]:
rating_valid_f.label.mean(), rating_test_f.label.mean() 

In [ ]:
rating_valid_f.columns

In [ ]:
import copy
def deal_with_each_u(x,u):
    items = np.array(x.iid)
    labels = np.array(x.label)
    titles = np.array(x.title)
    timestamp = np.array(x.timestamp)
    flags =  np.array(x.flag) 
    his = [0] # adding a '0' by default
    his_title = ['']
    results = []
    for i in range(items.shape[0]):
        results.append((u, items[i], timestamp[i], np.array(his), copy.copy(his_title),titles[i], labels[i], flags[i]))
        # training data
        if labels[i] > 0: # positive 
            his.append(items[i])
            his_title.append(titles[i])
    return results

In [ ]:
rating_train = rating_train.copy()

In [ ]:
rating_train['flag'] =  pd.DataFrame(np.ones(rating_train.shape[0])*-1, index=rating_train.index)
rating_valid_f['flag'] = pd.DataFrame(np.zeros(rating_valid_f.shape[0]), index=rating_valid_f.index)
rating_test_f['flag'] = pd.DataFrame(np.ones(rating_test_f.shape[0]), index=rating_test_f.index)
data = pd.concat([rating_train, rating_valid_f, rating_test_f],axis=0,ignore_index=True)
data = data.sort_values(by=['uid','timestamp'])
u_inter_all = data.groupby('uid').agg({'iid':list, 'label':list, 'title':list, 'timestamp':list,'flag':list})


In [ ]:
data.flag.unique()

In [ ]:
results = []
for u in u_inter_all.index:
    results.extend(deal_with_each_u(u_inter_all.loc[u],u))

In [ ]:
u_, i_, time_, label_, his_, his_title, title_,flag_ = [],[],[],[],[],[],[],[]
for re_ in results:
    u_.append(re_[0])
    i_.append(re_[1])
    time_.append(re_[2])
    his_.append(re_[3])
    his_title.append(re_[4])
    title_.append(re_[5])
    label_.append(re_[6])
    flag_.append(re_[7])

In [ ]:
data = pd.DataFrame({"uid":u_,'iid':i_,'label':label_, 'timestamp': time_ , 'his':his_,'his_title':his_title,'title':title_, 'flag': flag_})


In [ ]:
data.head(2)

In [ ]:
users = data.uid.unique()
items = data.iid.unique()
users_map = dict(zip(users, np.arange(users.shape[0])+1))
items_map = dict(zip(items, np.arange(items.shape[0])+1))

In [ ]:
users_map[0]=0
items_map[0]=0

In [ ]:
data['uid'] = data['uid'].map(users_map)
data['iid'] = data['iid'].map(items_map)
data.head(2)

In [ ]:
data['his'] = data['his'].apply(lambda x: [items_map[k] for k in x])

In [ ]:
data.label.describe()

In [ ]:
data.tail(5)

In [ ]:
train_ = data[data['flag'].isin([-1])].copy()
valid_ = data[data['flag'].isin([0])].copy()
test_ = data[data['flag'].isin([1])].copy()
train_.shape,valid_.shape,test_.shape

In [ ]:
train_user = train_['uid'].unique()
train_item = train_['iid'].unique()
valid_['not_cold'] = valid_[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")
test_['not_cold'] = test_[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")

In [ ]:
train_['not_cold'] = pd.DataFrame(np.ones(train_.shape[0]),index=train_.index).astype("int")
train_.head(2)

In [ ]:
save_path = "/data/zyang/datasets/ml-1m/"
train_.to_pickle(save_path+"train_ood2.pkl")
valid_.to_pickle(save_path+"valid_ood2.pkl")
test_.to_pickle(save_path+"test_ood2.pkl")

In [ ]:
valid_small = valid_.sample(frac=0.5,random_state=2023)
valid_small.to_pickle(save_path+"valid_small_ood2.pkl")

In [ ]:
valid_small.shape

In [ ]:
train_['his'].apply(len).describe()

In [ ]:
train_.label.mean(), test_.label.mean(), valid_.label.mean()  

In [ ]:
test_.groupby("uid").agg({"label":['count','sum']}).describe()

In [ ]:
train_.groupby("iid").agg({"label":['count','sum']}).describe()

In [ ]:
valid_[valid_['uid'].isin(train_['uid'].unique())].uid.unique().shape

In [ ]:
train_['iid'].unique()

In [ ]:
valid_[valid_['iid'].isin(train_['iid'].unique())].iid.unique().shape

In [ ]:
valid_.iid.unique().shape, test_.iid.unique().shape

In [ ]:
valid_.uid.unique().shape, test_.uid.unique().shape

In [ ]:
train_.uid.unique().shape, train_.iid.unique().shape

In [ ]:
m = train_.groupby('uid').agg({'label':'count'}).sort_values('label').reset_index()
m.plot(x='uid',kind='bar')

In [ ]:
train_.uid.max()



In [ ]:
n = valid_.groupby('uid').agg({'label':'count'})
idx = []
a = valid_.uid.unique()
for x in m['uid'].values[:]:
    if x in a:
        idx.append(x)
n.loc[idx].reset_index().plot(x='uid',kind='bar')

In [ ]:
n = test_.groupby('uid').agg({'label':'count'})
idx = []
a = test_.uid.unique()
for x in m['uid'].values[:]:
    if x in a:
        idx.append(x)
n.loc[idx].reset_index().plot(x='uid',kind='bar')

In [ ]:
save_path = "/data/zyang/datasets/ml-1m/"
train_ = pd.read_pickle(save_path+"train_ood2.pkl")
valid_ = pd.read_pickle(save_path+"valid_ood2.pkl")
test_ = pd.read_pickle(save_path+"test_ood2.pkl")

In [ ]:
max(train_.uid.max(),test_.uid.max(),valid_.uid.max())+1, max(train_.iid.max(),test_.iid.max(),valid_.iid.max())+1


In [ ]:
valid_small = pd.read_pickle(save_path+"valid_small_ood2.pkl")

In [ ]:
valid_small.uid.max(), valid_small.iid.max()

In [ ]:
import pandas as pd
save_path = "/data/zyang/datasets/ml-1m/"
# trai.to_pickle(save_path+"train_ood2.pkl")
valid_ = pd.read_pickle(save_path+"valid_ood2.pkl")
test_ = pd.read_pickle(save_path+"test_ood2.pkl")

In [ ]:
valid_.not_cold.mean()